<a href="https://colab.research.google.com/github/NarendraRaoJami/J_V_Narendra_Rao_Internship_2026_College/blob/main/J_V_Narendra_Rao/DuQuant_BERT_Benchmark.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Device Specifications

| Specification         | Details                                      |
|-----------------------|----------------------------------------------|
| **Platform**          | Google Colaboratory                          |
| **Runtime Type**      | GPU                                          |
| **Accelerator**       | NVIDIA Tesla T4                              |
| **GPU Memory**        | 15 GB GDDR6                                  |
| **CUDA Cores**        | 2,560                                        |
| **Tensor Cores**      | 320 (2nd Gen)                                |
| **GPU Architecture**  | Turing (SM 7.5)                              |
| **CUDA Version**      | 12.2                                         |
| **CPU**               | Intel Xeon (2 vCPUs)                         |
| **System RAM**        | ~12.7 GB                                     |
| **Disk Space**        | ~78 GB                                       |
| **Python Version**    | 3.10.x                                       |
| **OS**                | Ubuntu 22.04 LTS (64-bit)                    |
| **Driver Version**    | 525.xx (NVIDIA)                              |

In [1]:
!pip install -q transformers datasets evaluate accelerate pynvml scikit-learn scipy sentencepiece

In [2]:
import time, gc, warnings, threading, tempfile, os, copy
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from datasets import load_dataset
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
from scipy.stats import pearsonr, spearmanr

warnings.filterwarnings('ignore')
os.environ['TOKENIZERS_PARALLELISM'] = 'false'
torch.set_num_threads(2)
DEVICE       = 'cuda' if torch.cuda.is_available() else 'cpu'
BATCH_SIZE   = 32
MAX_SAMPLES  = 2000
MAX_LENGTH   = 128
POLL_INTERVAL_S = 0.005
print('Device:', DEVICE)

/usr/local/lib/python3.12/dist-packages/torch/cuda/__init__.py:65: FutureWarning: The pynvml package is deprecated. Please install nvidia-ml-py instead. If you did not install pynvml directly, please report this to the maintainers of the package that installed pynvml for you.
  import pynvml  # type: ignore[import]


Device: cuda


In [ ]:
MODEL_REGISTRY = {
    'SST2': {
        'BERT-base':  'textattack/bert-base-uncased-SST-2',
        'DistilBERT': 'distilbert-base-uncased-finetuned-sst-2-english',
        'AlBERT':     'Alireza1044/albert-base-v2-sst2',
        'MobileBERT': 'Alireza1044/mobilebert_sst2',
        'TinyBERT':   'M-FAC/bert-tiny-finetuned-sst2',
        'GPT-2':      'tanganke/gpt2_sst2',
    },
    'QNLI': {
        'BERT-base':  'textattack/bert-base-uncased-QNLI',
        'DistilBERT': 'textattack/distilbert-base-uncased-QNLI',
        'AlBERT':     'textattack/albert-base-v2-QNLI',
        'MobileBERT': 'Alireza1044/mobilebert_qnli',
        'TinyBERT':   'M-FAC/bert-tiny-finetuned-qnli',
        'GPT-2':      'tanganke/gpt2_qnli',
    },
    'MNLI': {
        'BERT-base':  'chromeNLP/textattack_bert_base_MNLI_fixed',
        'DistilBERT': 'typeform/distilbert-base-uncased-mnli',  
        'AlBERT':     'Alireza1044/albert-base-v2-mnli',
        'MobileBERT': 'Alireza1044/mobilebert_mnli',
        'TinyBERT':   'M-FAC/bert-tiny-finetuned-mnli',
        'GPT-2':      'tanganke/gpt2_mnli',
    },
    'QQP': {
        'BERT-base':  'textattack/bert-base-uncased-QQP',
        'DistilBERT': 'nyu-mll/distilbert-base-uncased-finetuned-qqp',
        'AlBERT':     'Alireza1044/albert-base-v2-qqp',
        'MobileBERT': 'Alireza1044/mobilebert_qqp',
        'TinyBERT':   'M-FAC/bert-tiny-finetuned-qqp',
        'GPT-2':      'tanganke/gpt2_qqp',
    },
    'RTE': {
        'BERT-base':  'textattack/bert-base-uncased-RTE',
        'DistilBERT': 'textattack/distilbert-base-uncased-RTE',
        'AlBERT':     'textattack/albert-base-v2-RTE',
        'MobileBERT': 'Alireza1044/mobilebert_rte',
        'TinyBERT':   'muhtasham/bert-tiny-finetuned-glue-rte',
        'GPT-2':      'tanganke/gpt2_rte',
    },
    'MRPC': {
        'BERT-base':  'textattack/bert-base-uncased-MRPC',
        'DistilBERT': 'textattack/distilbert-base-uncased-MRPC',
        'AlBERT':     'Alireza1044/albert-base-v2-mrpc',
        'MobileBERT': 'Alireza1044/mobilebert_mrpc',
        'TinyBERT':   'M-FAC/bert-tiny-finetuned-mrpc',
        'GPT-2':      'tanganke/gpt2_mrpc',
    },
    'CoLA': {
        'BERT-base':  'textattack/bert-base-uncased-CoLA',
        'DistilBERT': 'textattack/distilbert-base-uncased-CoLA',
        'AlBERT':     'Alireza1044/albert-base-v2-cola',
        'MobileBERT': 'Alireza1044/mobilebert_cola',
        'TinyBERT':   'howey/bert-base-uncased-cola',
        'GPT-2':      'tanganke/gpt2_cola',
    },
    'STS-B': {
        'BERT-base':  'textattack/bert-base-uncased-STS-B',
        'DistilBERT': 'assemblyai/distilbert-base-uncased-sst2',
        'AlBERT':     'Alireza1044/albert-base-v2-stsb',
        'MobileBERT': 'Alireza1044/mobilebert_stsb',
        'TinyBERT':   'M-FAC/bert-tiny-finetuned-stsb',
        'GPT-2':      'PavanNeerudu/gpt2-finetuned-stsb',
    },
    'WNLI': {
        'BERT-base':  'T-Systems-onsite/bert-base-uncased-wnli',
        'DistilBERT': 'textattack/distilbert-base-uncased-WNLI',
        'AlBERT':     'Alireza1044/albert-base-v2-wnli',
        'MobileBERT': 'T-Systems-onsite/bert-base-uncased-wnli',
        'TinyBERT':   'muhtasham/bert-tiny-finetuned-glue-rte',
        'GPT-2':      'tanganke/gpt2_rte',
    },
}

DATASETS = {
    'SST2':  ('stanfordnlp/sst2',  None,    'validation',         'sentence',                'label'),
    'QNLI':  ('nyu-mll/glue',      'qnli',  'validation',         ('question','sentence'),   'label'),
    'MNLI':  ('nyu-mll/glue',      'mnli',  'validation_matched', ('premise','hypothesis'),  'label'),
    'QQP':   ('nyu-mll/glue',      'qqp',   'validation',         ('question1','question2'), 'label'),
    'RTE':   ('nyu-mll/glue',      'rte',   'validation',         ('sentence1','sentence2'), 'label'),
    'MRPC':  ('nyu-mll/glue',      'mrpc',  'validation',         ('sentence1','sentence2'), 'label'),
    'CoLA':  ('nyu-mll/glue',      'cola',  'validation',         'sentence',                'label'),
    'STS-B': ('nyu-mll/glue',      'stsb',  'validation',         ('sentence1','sentence2'), 'label'),
    'WNLI':  ('nyu-mll/glue',      'wnli',  'validation',         ('sentence1','sentence2'), 'label'),
}

MODEL_FAMILIES   = ['BERT-base', 'DistilBERT', 'AlBERT', 'MobileBERT', 'TinyBERT', 'GPT-2']
REGRESSION_TASKS = {'STS-B'}

LABEL_REMAPS = {
    ('MNLI', 'AlBERT'):     {0: 2, 1: 0, 2: 1},
    ('MNLI', 'MobileBERT'): {0: 2, 1: 0, 2: 1},
    ('MNLI', 'TinyBERT'):   {0: 2, 1: 0, 2: 1},
    ('MNLI', 'GPT-2'):      {0: 2, 1: 0, 2: 1},
}

INT8_MODELS = {'AlBERT'}

print(f'Datasets : {list(DATASETS.keys())}')
print(f'Models   : {MODEL_FAMILIES}')


Datasets : ['SST2', 'QNLI', 'MNLI', 'QQP', 'RTE', 'MRPC', 'CoLA', 'STS-B']
Models   : ['BERT-base', 'DistilBERT', 'AlBERT', 'MobileBERT', 'TinyBERT', 'GPT-2']


In [ ]:
from huggingface_hub import HfApi
import requests

api = HfApi()
bad_ids = []

print("Validating all checkpoint IDs on HuggingFace Hub...")
for ds_name, model_map in MODEL_REGISTRY.items():
    for model_name, hf_id in model_map.items():
        try:
            url = f"https://huggingface.co/{hf_id}/resolve/main/config.json"
            r = requests.head(url, timeout=10, allow_redirects=True)
            if r.status_code == 200:
                print(f"  OK   {ds_name}/{model_name}: {hf_id}")
            else:
                print(f"  FAIL {ds_name}/{model_name}: {hf_id}  [HTTP {r.status_code}]")
                bad_ids.append((ds_name, model_name, hf_id, r.status_code))
        except Exception as e:
            print(f"  ERR  {ds_name}/{model_name}: {hf_id}  [{e}]")
            bad_ids.append((ds_name, model_name, hf_id, str(e)))

if bad_ids:
    print(f"\n*** {len(bad_ids)} FAILED IDs — fix before running benchmark ***")
    for ds, mdl, hid, code in bad_ids:
        print(f"  {ds}/{mdl}: {hid}  -> {code}")
else:
    print(f"\nAll {sum(len(v) for v in MODEL_REGISTRY.values())} checkpoints verified OK.")


Validating all checkpoint IDs on HuggingFace Hub...
  OK   SST2/BERT-base: textattack/bert-base-uncased-SST-2
  OK   SST2/DistilBERT: distilbert-base-uncased-finetuned-sst-2-english
  OK   SST2/AlBERT: Alireza1044/albert-base-v2-sst2
  OK   SST2/MobileBERT: Alireza1044/mobilebert_sst2
  OK   SST2/TinyBERT: M-FAC/bert-tiny-finetuned-sst2
  OK   SST2/GPT-2: tanganke/gpt2_sst2
  OK   QNLI/BERT-base: textattack/bert-base-uncased-QNLI
  OK   QNLI/DistilBERT: textattack/distilbert-base-uncased-QNLI
  OK   QNLI/AlBERT: Alireza1044/albert-base-v2-qnli
  OK   QNLI/MobileBERT: Alireza1044/mobilebert_qnli
  OK   QNLI/TinyBERT: M-FAC/bert-tiny-finetuned-qnli
  OK   QNLI/GPT-2: tanganke/gpt2_qnli
  OK   MNLI/BERT-base: JeremiahZ/bert-base-uncased-mnli
  OK   MNLI/DistilBERT: textattack/distilbert-base-uncased-MNLI
  OK   MNLI/AlBERT: Alireza1044/albert-base-v2-mnli
  OK   MNLI/MobileBERT: Alireza1044/mobilebert_mnli
  OK   MNLI/TinyBERT: M-FAC/bert-tiny-finetuned-mnli
  OK   MNLI/GPT-2: tanganke/gp

In [5]:
# ── Power / energy sampler ────────────────────────────────────────────────────
try:
    import pynvml
    pynvml.nvmlInit()
    _nvml_handle = pynvml.nvmlDeviceGetHandleByIndex(0)
    NVML_AVAILABLE = True
    print('pynvml ready |', pynvml.nvmlDeviceGetName(_nvml_handle))
except Exception as e:
    NVML_AVAILABLE = False
    print(f'pynvml unavailable ({e}) — energy reported as 0')

class PowerSampler:
    def __init__(self):
        self._samples, self._running, self._thread = [], False, None
    def _poll(self):
        while self._running:
            if NVML_AVAILABLE:
                try: self._samples.append(pynvml.nvmlDeviceGetPowerUsage(_nvml_handle))
                except: pass
            time.sleep(POLL_INTERVAL_S)
    def start(self):
        self._samples = []; self._running = True
        self._thread = threading.Thread(target=self._poll, daemon=True)
        self._thread.start()
    def stop(self):
        self._running = False; self._thread.join(timeout=0.5)
        return float(np.mean(self._samples)) if self._samples else 0.0

pynvml ready | Tesla T4


In [ ]:
import torch
import torch.nn as nn

CLIP_PERCENTILE = 99.9   
GROUP_SIZE      = 64     

def quantize_int4_groupwise(W: torch.Tensor,
                             group_size: int = GROUP_SIZE,
                             clip_pct: float = CLIP_PERCENTILE) -> torch.Tensor:

    W = W.float()
    out_ch, in_ch = W.shape

    if group_size <= 0 or group_size >= in_ch:
        threshold = torch.quantile(W.abs().reshape(out_ch, -1),
                                   clip_pct / 100.0, dim=1, keepdim=True).clamp(min=1e-8)
        W_clip = W.clamp(-threshold, threshold)
        scale  = threshold / 7.0
        return torch.clamp(torch.round(W_clip / scale), -8, 7) * scale

    pad = (group_size - in_ch % group_size) % group_size
    if pad:
        W = torch.cat([W, W.new_zeros(out_ch, pad)], dim=1)

    n_groups = W.shape[1] // group_size
    Wg = W.reshape(out_ch * n_groups, group_size)

    threshold = torch.quantile(Wg.abs(), clip_pct / 100.0, dim=1, keepdim=True).clamp(min=1e-8)
    Wg_clip   = Wg.clamp(-threshold, threshold)
    scale     = threshold / 7.0
    Wg_q      = torch.clamp(torch.round(Wg_clip / scale), -8, 7) * scale

    W_q = Wg_q.reshape(out_ch, -1)
    if pad:
        W_q = W_q[:, :in_ch]
    return W_q


def quantize_model_int4(model: nn.Module,
                         group_size: int = GROUP_SIZE,
                         clip_pct: float = CLIP_PERCENTILE) -> nn.Module:
    model.cpu()
    with torch.no_grad():
        for module in model.modules():
            if isinstance(module, nn.Linear):
                W = module.weight.data
                module.weight.data = quantize_int4_groupwise(
                    W, group_size=group_size, clip_pct=clip_pct
                ).to(W.dtype)
    return model


print(f'Quantization ready | group_size={GROUP_SIZE}, clip_pct={CLIP_PERCENTILE}')


def quantize_model_int8(model: nn.Module) -> nn.Module:
    model.cpu()
    quantized = torch.quantization.quantize_dynamic(
        model, {nn.Linear}, dtype=torch.qint8
    )
    return quantized


Quantization ready | group_size=64, clip_pct=99.9


In [ ]:
# ── Utilities ─────────────────────────────────────────────────────────────────

def memory_mb(model: nn.Module) -> float:
    with tempfile.NamedTemporaryFile(delete=False, suffix='.pt') as f:
        torch.save(model.state_dict(), f.name)
        size = os.path.getsize(f.name) / (1024 * 1024)
    os.remove(f.name)
    return round(size, 2)


def tokenize_batch(tok, texts, device):
    is_pair = type(texts[0]) in (tuple, list)
    if is_pair:
        enc = tok([t[0] for t in texts], [t[1] for t in texts],
                  truncation=True, padding=True, max_length=MAX_LENGTH, return_tensors='pt')
    else:
        enc = tok(texts, truncation=True, padding=True,
                  max_length=MAX_LENGTH, return_tensors='pt')
    return {k: v.to(device) for k, v in enc.items()}


def benchmark(model: nn.Module, tok, ds_cfg, is_regression: bool = False, label_map: dict = None) -> dict:

    path, config, split, text_col, label_col = ds_cfg
    ds = load_dataset(path, config, split=split) if config else load_dataset(path, split=split)
    ds = ds.select(range(min(MAX_SAMPLES, len(ds))))
    n  = len(ds)

    is_pair      = type(text_col) in (tuple, list)
    model_device = next(model.parameters()).device
    model.eval()

    preds, labels, latencies, energies = [], [], [], []
    t_total = time.perf_counter()

    for i in range(0, n, BATCH_SIZE):
        batch   = ds[i: i + BATCH_SIZE]
        texts   = list(zip(batch[text_col[0]], batch[text_col[1]])) if is_pair else batch[text_col]
        enc     = tokenize_batch(tok, texts, model_device)
        batch_n = len(batch[label_col])

        sampler = PowerSampler(); sampler.start()
        t0 = time.perf_counter()
        with torch.no_grad():
            out = model(**enc)
        if model_device.type == 'cuda':
            torch.cuda.synchronize()
        elapsed_ms   = (time.perf_counter() - t0) * 1000
        avg_power_mw = sampler.stop()

        latencies.append(elapsed_ms / batch_n)
        energies.append((avg_power_mw / 1000.0) * (elapsed_ms / 1000.0) * 1000.0 / batch_n)

        if is_regression:
            preds.extend(out.logits.squeeze(-1).cpu().tolist())
        else:
            raw = out.logits.argmax(-1).cpu().tolist()
            if label_map:
                raw = [label_map.get(p, p) for p in raw]
            preds.extend(raw)
        labels.extend(batch[label_col])

    throughput = n / (time.perf_counter() - t_total)
    lat = float(np.mean(latencies))
    eng = float(np.mean(energies))

    if is_regression:
        p = np.array(preds, dtype=float)
        l = np.array(labels, dtype=float)
        p = np.clip(p, 0.0, 5.0)
        pearson_r  = round(float(pearsonr(p, l)[0])  * 100, 2)
        spearman_r = round(float(spearmanr(p, l)[0]) * 100, 2)
        return {'Accuracy': np.nan, 'Precision': np.nan, 'Recall': np.nan, 'F1': np.nan,
                'Pearson': pearson_r, 'Spearman': spearman_r,
                'Latency (ms)': round(lat, 4), 'Throughput (sps)': round(throughput, 1),
                'Energy (mJ)': round(eng, 4)}

    acc  = accuracy_score(labels, preds)
    prec = precision_score(labels, preds, average='weighted', zero_division=0)
    rec  = recall_score(labels, preds, average='weighted', zero_division=0)
    f1   = f1_score(labels, preds, average='weighted', zero_division=0)
    return {'Accuracy': round(acc*100, 2), 'Precision': round(prec*100, 2),
            'Recall': round(rec*100, 2), 'F1': round(f1*100, 2),
            'Pearson': np.nan, 'Spearman': np.nan,
            'Latency (ms)': round(lat, 4), 'Throughput (sps)': round(throughput, 1),
            'Energy (mJ)': round(eng, 4)}


print('Utilities defined.')


Utilities defined.


In [ ]:
results   = []
completed = set()

for ds_name, ds_cfg in DATASETS.items():
    # ── Determine task type ───────────────────────────────────────────────────
    is_regression = ds_name in REGRESSION_TASKS

    for model_name in MODEL_FAMILIES:

        if (ds_name, model_name) in completed:
            print(f'Skip (done): {ds_name} | {model_name}'); continue

        hf_id = MODEL_REGISTRY.get(ds_name, {}).get(model_name)
        if hf_id is None:
            print(f'No checkpoint for {ds_name} | {model_name} — skipping.'); continue

        label_map = LABEL_REMAPS.get((ds_name, model_name), None)

        print(f"\n{'='*60}")
        print(f'  Dataset: {ds_name}  |  Model: {model_name}')
        print(f'  Checkpoint: {hf_id}')
        print(f'  Task type: {"regression (Pearson/Spearman)" if is_regression else "classification (Accuracy)"}')
        print(f"{'='*60}")

        tok = fp32_model = q_model = None
        try:
            print('  [1/3] Loading checkpoint...')
            tok = AutoTokenizer.from_pretrained(hf_id)

            if tok.pad_token is None:
                tok.pad_token    = tok.eos_token
                tok.padding_side = 'left'

            fp32_model = AutoModelForSequenceClassification.from_pretrained(hf_id)
            fp32_model = fp32_model.to(DEVICE)

            if fp32_model.config.pad_token_id is None:
                fp32_model.config.pad_token_id = tok.pad_token_id

            print('  [2/3] FP32 baseline...')
            fp32_mem = memory_mb(fp32_model)
            fp32_m   = benchmark(fp32_model, tok, ds_cfg, is_regression, label_map)

            if is_regression:
                acc_str = f"Pearson={fp32_m['Pearson']:.1f}  Spearman={fp32_m['Spearman']:.1f}"
            else:
                acc_str = f"Acc={fp32_m['Accuracy']:.1f}%"
            print(f'  FP32: {acc_str}, Latency={fp32_m["Latency (ms)"]:.2f}ms/sample')

            results.append({'Dataset': ds_name, 'Model': model_name, 'Method': 'FP32-baseline',
                            'Bits': 32, 'Memory (MB)': fp32_mem, **fp32_m,
                            'Accuracy Delta': np.nan, 'Pearson Delta': np.nan})

            use_int8 = model_name in INT8_MODELS
            quant_label = 'DuQuant-INT8' if use_int8 else 'DuQuant-INT4'
            bits = 8 if use_int8 else 4
            print(f'  [3/3] {quant_label} quantization + benchmark...')
            q_model = copy.deepcopy(fp32_model)
            if use_int8:
                q_model = quantize_model_int8(q_model)
            else:
                q_model = quantize_model_int4(q_model).to(DEVICE)

            q_mem = memory_mb(q_model)
            q_m   = benchmark(q_model, tok, ds_cfg, is_regression, label_map)

            if is_regression:
                delta_acc = np.nan
                delta_pearson = round(q_m['Pearson'] - fp32_m['Pearson'], 2)                                 if not (np.isnan(q_m['Pearson']) or np.isnan(fp32_m['Pearson'])) else np.nan
                q_acc_str = (f"Pearson={q_m['Pearson']:.1f} (Δ={delta_pearson:+.1f})  "
                             f"Spearman={q_m['Spearman']:.1f}")
            else:
                delta_acc = round(q_m['Accuracy'] - fp32_m['Accuracy'], 2)                             if not (np.isnan(q_m['Accuracy']) or np.isnan(fp32_m['Accuracy'])) else np.nan
                delta_pearson = np.nan
                q_acc_str = f"Acc={q_m['Accuracy']:.1f}% (Δ={delta_acc:+.1f}%)"

            print(f'  INT4: {q_acc_str}, Latency={q_m["Latency (ms)"]:.2f}ms/sample')

            results.append({'Dataset': ds_name, 'Model': model_name, 'Method': quant_label,
                            'Bits': bits, 'Memory (MB)': q_mem, **q_m,
                            'Accuracy Delta': delta_acc, 'Pearson Delta': delta_pearson})

            completed.add((ds_name, model_name))

        except Exception as e:
            import traceback
            print(f'  ERROR: {e}'); traceback.print_exc()
        finally:
            for obj in [fp32_model, q_model, tok]:
                try: del obj
                except: pass
            gc.collect()
            if DEVICE == 'cuda': torch.cuda.empty_cache()

print('\n=== Benchmarking complete ===')



  Dataset: SST2  |  Model: BERT-base
  Checkpoint: textattack/bert-base-uncased-SST-2
  Task type: classification (Accuracy)
  [1/3] Loading checkpoint...


config.json:   0%|          | 0.00/477 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

...

Loading weights:   0%|          | 0/149 [00:00<?, ?it/s]

GPT2ForSequenceClassification LOAD REPORT from: PavanNeerudu/gpt2-finetuned-stsb
Key                                     | Status     |  | 
----------------------------------------+------------+--+-
transformer.h.{0...11}.attn.masked_bias | UNEXPECTED |  | 
transformer.h.{0...11}.attn.bias        | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


  [2/3] FP32 baseline...


model.safetensors:   0%|          | 0.00/510M [00:00<?, ?B/s]

  FP32: Pearson=3.8  Spearman=4.6, Latency=2.92ms/sample
  [3/3] INT4 quantization + benchmark...
  INT4: Pearson=3.9 (Δ=+0.1)  Spearman=4.4, Latency=3.04ms/sample

=== Benchmarking complete ===


## Results

In [13]:
df = pd.DataFrame(results)
pd.set_option('display.max_columns', None)
pd.set_option('display.width', 240)
pd.set_option('display.float_format', '{:.2f}'.format)

# Classification columns
clf_cols  = ['Model', 'Method', 'Bits', 'Memory (MB)',
             'Accuracy', 'F1', 'Accuracy Delta',
             'Latency (ms)', 'Throughput (sps)', 'Energy (mJ)']

# Regression columns (STS-B)
reg_cols  = ['Model', 'Method', 'Bits', 'Memory (MB)',
             'Pearson', 'Spearman', 'Pearson Delta',
             'Latency (ms)', 'Throughput (sps)', 'Energy (mJ)']

display_df = df.copy()

for ds in display_df['Dataset'].unique():
    print(f'\n{" = "*35}')
    print(f'  {ds}')
    print(f'{" = "*35}')
    sub = display_df[display_df['Dataset'] == ds]
    use_cols = reg_cols if ds in REGRESSION_TASKS else clf_cols
    print(sub[[c for c in use_cols if c in display_df.columns]].to_string(index=False))



 =  =  =  =  =  =  =  =  =  =  =  =  =  =  =  =  =  =  =  =  =  =  =  =  =  =  =  =  =  =  =  =  =  =  = 
  SST2
 =  =  =  =  =  =  =  =  =  =  =  =  =  =  =  =  =  =  =  =  =  =  =  =  =  =  =  =  =  =  =  =  =  =  = 
     Model        Method  Bits  Memory (MB)  Accuracy (%)  F1 (%)  Accuracy Delta  Latency (ms)  Throughput (sps)  Energy (mJ)
 BERT-base FP32-baseline    32       417.73         92.43   92.43             NaN          2.89            299.00       162.41
 BERT-base  DuQuant-INT4     4       417.73         92.43   92.43            0.00          2.30            369.20       145.53
DistilBERT FP32-baseline    32       255.45         91.06   91.05             NaN          1.15            684.00        73.94
DistilBERT  DuQuant-INT4     4       255.45         90.71   90.71           -0.35          1.15            664.80        70.79
    AlBERT FP32-baseline    32        44.59         92.32   92.32             NaN          3.08            292.30       203.46
    AlBERT  DuQuan